# E2 — MoG40 (2D): 40 Gaussian modes, generic annulus jumps, closed-form score

$$\boxed{V(x) = -\frac1\beta \log\sum_{k=1}^{40}\exp\!\Big(-\tfrac12\|x-\mu_k\|^2\Big)}\quad\Longrightarrow\quad \pi(x)\propto e^{-\beta V} = \sum_k e^{-\|x-\mu_k\|^2/2},$$
an equal-weight mixture of $\mathcal N(\mu_k, I_2)$. **The $1/\beta$ prefactor is essential** — it is what makes the barriers right:
$$\nabla V(x) = \frac1\beta\sum_k \omega_k(x)\,(x-\mu_k),\qquad \omega_k = \mathrm{softmax}_k(-\tfrac12\|x-\mu_k\|^2),$$
and the effective barrier between modes at distance $d$ is $\beta\Delta V = \tfrac{d^2}{8} - \log 2$ — heterogeneous by design.

**Neither PT nor LSC-CP receives mode locations.** The jump law is deliberately generic: $r = \rho\,u_\phi$ with $\rho\sim\mathrm{Unif}[4,15]$, $\phi\sim\mathrm{Unif}[0,2\pi)$, the interval $[4,15]$ set from the NN-distance histogram alone (printed below). The Lévy score for this law has a **closed form with zero potential evaluations**.

In [1]:
EXPERIMENT = "mog40"
import os, sys, math, time, json
sys.path.insert(0, os.path.abspath(".."))
from src.gpu_guard import select_gpu
select_gpu(int(os.environ.get("JCP_GPU", "4")))
import torch
assert torch.cuda.device_count() == 1, "GPU guard must mask to exactly one device"
torch.set_default_dtype(torch.float64)
import numpy as np
import pandas as pd

from src import config as C
from src.experiments import build_e2, make_sampler_factory, make_metrics
from src.runner import (run_experiment, run_one, refine_dt, quadrature_refinement,
                        write_timeseries_csv, write_summary_csv, write_manifest,
                        ula_first_passage, hardware_manifest)
from src.samplers import tune_ladder
from src.certificate import make_phi_family, certificate_grid, certificate_importance
from src.plotting import make_all_figures, apply_style

DEV = "cuda"
RESULTS = os.path.abspath(os.path.join("..", "results", EXPERIMENT))
FIGURES = os.path.abspath(os.path.join("..", "figures", EXPERIMENT))
os.makedirs(RESULTS, exist_ok=True); os.makedirs(FIGURES, exist_ok=True)
exp = build_e2(device=DEV)
cfg = exp.cfg
print(f"experiment={cfg.name}  d={cfg.d}  N={cfg.n_particles}  T={cfg.T}  dt0={cfg.dt}")
print(f"beta={cfg.beta}  eps={cfg.eps}  lambda={cfg.lam}  seeds={cfg.seeds}")
print(hardware_manifest())

experiment=mog40  d=2  N=2500  T=100.0  dt0=0.01
beta=8.0  eps=0.125  lambda=1.0  seeds=(0, 1, 2, 3, 4)


{'cpu': 'AMD EPYC 9554 64-Core Processor', 'gpu': 'NVIDIA H200 NVL', 'torch': '2.13.0+cu130', 'cuda': '13.0', 'python': '3.12.13', 'git_sha': 'b84ad342ffc64e1d0a657de15c492269bca0e65c', 'cuda_visible_devices': '5', 'gpu_compute_apps_at_start': ['GPU-c8768979-ae7d-8ca1-a33d-2874c9d21fae, 1694731, 28790 MiB', 'GPU-e003b5e1-3cce-0c27-5a6b-9f594c1b7682, 1694732, 29184 MiB', 'GPU-e003b5e1-3cce-0c27-5a6b-9f594c1b7682, 1885830, 710 MiB', 'GPU-0693e45d-5be4-82d1-e0fa-e72261e24e3b, 1597887, 108278 MiB', 'GPU-0693e45d-5be4-82d1-e0fa-e72261e24e3b, 1766636, 814 MiB', 'GPU-0693e45d-5be4-82d1-e0fa-e72261e24e3b, 1859671, 1752 MiB', 'GPU-b7477750-5992-8ae3-fde2-11bf70934062, 1694733, 29174 MiB', 'GPU-b7477750-5992-8ae3-fde2-11bf70934062, 1727069, 814 MiB', 'GPU-b7477750-5992-8ae3-fde2-11bf70934062, 1853338, 7582 MiB']}


## The model

$\mu_k \sim \mathrm{Unif}([-40,40]^2)$ frozen from `np.random.default_rng(0)` and saved to CSV. $N=2500$, $T=100$, $\Delta t_0 = 0.01$, box $[-65,65]^2$, all particles initialised at $\mu_0 + 0.5\,\xi$. Partition ($K=40$): nearest-mode Voronoi with $p^\star_k = 1/40$ (exact up to $O(e^{-d^2/8})$ Voronoi leakage, absorbed into the occupancy bias floor). Reference: exact i.i.d. mixture draws.

Barrier verification below uses the nearest-neighbour gap of mode 0 and a 1D Kramers estimate along the inter-mode line ($\omega_{\min} = \omega_{\rm saddle} \approx 1$ for unit-variance components).

In [2]:
np.savetxt(os.path.join(RESULTS, "modes.csv"), exp.pot.mu.cpu().numpy(),
           delimiter=",", header="mu_x,mu_y", comments="")
dists = torch.cdist(exp.pot.mu, exp.pot.mu)
dists.fill_diagonal_(float("inf"))
nn = dists.min(dim=1).values.cpu().numpy()
print("NN-distance histogram (the sole input to the jump interval [4,15]):")
hist, edges = np.histogram(nn, bins=[0, 2, 4, 6, 8, 10, 12, 14, 16, 20])
for h, e0, e1 in zip(hist, edges[:-1], edges[1:]):
    print(f"  [{e0:>4.1f}, {e1:>4.1f}): {'#'*h} {h}")
print(f"NN distances: min {nn.min():.2f}, median {np.median(nn):.2f}, max {nn.max():.2f}")
print(f"beta*DeltaV across gaps d=4..16: {4**2/8 - math.log(2):.1f} .. {16**2/8 - math.log(2):.1f}")
print(f"mode-0 NN gap d0 = {exp.extras['nn_dist_mode0']:.2f}, "
      f"beta*dV = {exp.extras['beta_dV_mode0']:.2f}, Kramers tau ~ {exp.kramers_tau:.0f}")

g = torch.Generator(device=DEV); g.manual_seed(0)
barrier_report = ula_first_passage(exp.pot, exp.box, exp.init_fn(cfg.n_particles, g),
                                   exp.in_basin0, cfg.dt, int(cfg.T/cfg.dt), C.EPS, g)
barrier_report["kramers_tau_mode0"] = exp.kramers_tau
print("ULA first-passage out of Voronoi cell 0:", barrier_report)

NN-distance histogram (the sole input to the jump interval [4,15]):
  [ 0.0,  2.0): ## 2
  [ 2.0,  4.0): ##### 5
  [ 4.0,  6.0): ######### 9
  [ 6.0,  8.0): ######### 9
  [ 8.0, 10.0): ######### 9
  [10.0, 12.0): ## 2
  [12.0, 14.0): # 1
  [14.0, 16.0): ## 2
  [16.0, 20.0): # 1
NN distances: min 1.22, median 6.78, max 16.02
beta*DeltaV across gaps d=4..16: 1.3 .. 31.3
mode-0 NN gap d0 = 6.43, beta*dV = 4.48, Kramers tau ~ 552


ULA first-passage out of Voronoi cell 0: {'n_particles': 2500, 'T': 100.0, 'n_exits': 268, 'exit_fraction': 0.1072, 'mfpt_estimate': 883.6875, 'kramers_tau_mode0': 551.9750007500811}


## The closed-form Lévy score

With $d_k = x - \mu_k$, $m_k = u_\phi\cdot d_k$, the mixture gives
$$\frac{\pi(x-\theta\rho u)}{\pi(x)} = \sum_k \omega_k(x)\, e^{\theta\rho m_k - \theta^2\rho^2/2},\qquad \omega_k = \mathrm{softmax}_k(-\tfrac12\|d_k\|^2).$$
Completing the square, $I(\rho,m) = \int_0^1 e^{-\theta^2\rho^2/2+\theta\rho m}d\theta = \frac{\sqrt{\pi/2}}{\rho}e^{m^2/2}[\mathrm{erf}(\tfrac{\rho-m}{\sqrt2})+\mathrm{erf}(\tfrac{m}{\sqrt2})]$, and the $\rho$ from $r=\rho u$ cancels the $1/\rho$. With $F(z) = z\,\mathrm{erf}(z/\sqrt2)+\sqrt{2/\pi}e^{-z^2/2}$ ($F'=\mathrm{erf}(z/\sqrt2)$, $F$ even):
$$H(m) = \int_a^b \rho I(\rho,m)\,d\rho = \sqrt{\tfrac\pi2}e^{m^2/2}\underbrace{\big[F(b-m)-F(a-m)+(b-a)\mathrm{erf}(\tfrac m{\sqrt2})\big]}_{\mathcal B(m)>0},$$
so only the $\phi$ integral needs numerics (an $M_\phi$-point trapezoid rule — spectrally accurate for the periodic integrand):
$$S(x) \approx -\frac{\lambda}{M_\phi(b-a)}\sum_{\ell}u_\ell\sum_k \exp\big[\log\omega_k + m_{k\ell}^2/2\big]\sqrt{\tfrac\pi2}\,\mathcal B(m_{k\ell}).$$

**$\mathcal B$ must be evaluated by branches.** With $g(z) = e^{-z^2/2}[\sqrt{2/\pi}-z\,\mathrm{erfcx}(z/\sqrt2)]$ (so $F(z)=|z|+g(|z|)$), the $O(m)$ parts cancel *analytically* in the outer regimes:

| regime | formula |
|---|---|
| $m\ge b$ | $\mathcal B = g(m-b)-g(m-a)-(b-a)\,\mathrm{erfc}(m/\sqrt2)$ |
| $m\le 0$ | $\mathcal B = g(b-m)-g(a-m)+(b-a)\,\mathrm{erfc}(-m/\sqrt2)$ |
| $0<m<b$ | direct, safe |

The naive form has 100% relative error at $m=30$; the branched form $\sim10^{-12}$ (validated against 3000-digit mpmath, and the whole closed form against a brute-force 3-D quadrature at $10^{-8}$, `tests/test_score.py`). Our implementation additionally factors the dominant exponential out of each branch so $\log\mathcal B$ is finite on the whole real line, then applies the same log-space $(M, v)$ accumulation as the generic shell score.

The $\theta,\rho$ integrals being analytic, the only quadrature parameter is $M_\phi$; the §9.5 table in the run section records $\mathcal R$ and terminal metrics for $M_\phi \in \{16,32,64\}$ (production default 32, verified against 64). Note the *pointwise* direction-quadrature error at $M_\phi=32$ is $O(10^{-3})$ relative (far-mode terms peak in $\phi$ with width $\sim 1/\sqrt{b\,d_k}$), but its weak imprint against smooth test functions — which is what invariance is — integrates out, as the certificate shows.

In [3]:
DEFAULT_QUAD = dict(m_phi=C.M_PHI)
phis = make_phi_family(2, [0.0, 0.0], 30.0, DEV)

def cert_e2(m_phi):
    score = exp.make_score(m_phi=m_phi)
    shifts, logw = exp.law.quadrature_shifts(16, 64)   # fine continuous-nu J side
    return certificate_grid(exp.pot, score, shifts, logw, cfg.lam, cfg.beta,
                            phis, [-60.0, -60.0], [60.0, 60.0],
                            n_panels=120, nodes_per_panel=6, chunk=8192)

## Target preservation: the stationarity identity $(\star)$

The LSC-CP generator is
$$\mathcal A f = \big[-\nabla V + S_{\nu,\beta}\big]\cdot\nabla f + \varepsilon\,\Delta f + \lambda\!\int\!\big[f(x+r)-f(x)\big]\nu(dr),$$
with $\nu$ a **probability** measure and
$$S_{\nu,\beta}(x) = -\lambda \int \nu(dr)\; r \int_0^1 \exp\!\Big[-\beta\big(V(x-\theta r) - V(x)\big)\Big]\, d\theta .$$
Raw CP is the same generator with $S \equiv 0$.

Write $p = e^{-\beta V}/Z$. The overdamped part is $\pi$-reversible, so invariance of $\pi$ is equivalent to
$$\int S\cdot\nabla\varphi \, d\pi + \int J\varphi \, d\pi = 0 \qquad \forall\, \varphi \in C_c^\infty . \tag{$\star$}$$

**Jump term.** Shift the integration variable and apply the fundamental theorem of calculus along $\theta \mapsto y - \theta r$:
$$\int J\varphi\,d\pi = \lambda\!\int\!\nu(dr)\!\int\!\varphi(y)\big[p(y-r)-p(y)\big]dy = -\lambda\!\int\!\nu(dr)\!\int_0^1\!\!\int\!\varphi(y)\,r\!\cdot\!\nabla p(y-\theta r)\,dy\,d\theta.$$

**Drift term.** $S(x) = -\lambda\int\nu(dr)\,r\int_0^1 \frac{p(x-\theta r)}{p(x)}d\theta$ (identical to the boxed formula since $p \propto e^{-\beta V}$), so integrating by parts in $x$:
$$\int S\cdot\nabla\varphi\,d\pi = -\lambda\!\int\!\nu(dr)\!\int_0^1\!\!\int\!\big(r\!\cdot\!\nabla\varphi(x)\big) p(x-\theta r)\,dx\,d\theta = +\lambda\!\int\!\nu(dr)\!\int_0^1\!\!\int\!\varphi(x)\,r\!\cdot\!\nabla p(x-\theta r)\,dx\,d\theta.$$

The two cancel identically. **Target preservation is unconditional in $\nu$** — any finite-activity jump law works; only the *speed* depends on $\nu$.

### The measured certificate $\mathcal R(\varphi)$

For smooth bounded test functions (products of tanh ridges) we report
$$\mathcal R(\varphi) = \frac{\big|\int S_{\nu,\beta}\!\cdot\!\nabla\varphi\,d\pi + \int J_\nu\varphi\,d\pi\big|}{\big|\int J_\nu\varphi\,d\pi\big|},$$
zero in exact arithmetic; the measured value is the combined defect of the $\theta$/$\rho$ quadratures. Two implementation notes, both load-bearing:

1. **The integration domain extends a full jump length beyond the target's effective support.** Order-one contributions to $(\star)$ live where $\pi$ is tiny and $S$ is enormous; a deliberately tight box produces a large residual (demonstrated below and regression-tested).
2. **The drift integrand $p\,S\cdot\nabla\varphi$ is assembled in log space** from the score's $(M, v)$ parts as $\exp(-\beta V + M)\,v\cdot\nabla\varphi$: in linear fp64 arithmetic $p$ underflows exactly where $\|S\|$ is astronomical, silently dropping those order-one far-field contributions. The residual uses the *uncapped* $M$; the deployed drift caps $M$ at $M_{\max}=600$, but because taming saturates (the tamed step tends to $-v/\|v\|$), the deployed tamed step differs from the uncapped one by $O(e^{-M_{\max}})$ — that saturation defect is reported alongside $\mathcal R$ and is $\lesssim 10^{-250}$ here.

A useful exact identity (change of variables $x \to x+\theta_p r$ in the drift term): for the *implemented* quadrature score,
$$\int S\cdot\nabla\varphi\,d\pi + \int J\varphi\,d\pi \;=\; \lambda\,\mathbb E_\pi\!\int\!\nu(dr)\Big[\varphi(x+r)-\varphi(x) - \sum_p \hat w_p\, r\cdot\nabla\varphi(x+\theta_p r)\Big],$$
i.e. the residual is **independent of $V$** and equals the $\theta$-quadrature error on the smooth test-function integrand. This is why moderate pointwise errors of the Gauss–Legendre rule on the stiff factor $e^{\beta\Delta V}$ do not translate into a weak (distributional) defect of the sampled law.

In [4]:
cert_report = cert_e2(**DEFAULT_QUAD)
print("R(phi) on the generous box [-60,60]^2 (mode cloud [-40,40]^2 + 4 sigma + jump reach 15):")
for i in range(len(phis)):
    print(f"  phi_{i}: R = {cert_report[f'phi_{i}']['residual']:.3e}")
print(f"max R = {cert_report['max_residual']:.3e}")
assert cert_report["max_residual"] < 1e-6

R(phi) on the generous box [-60,60]^2 (mode cloud [-40,40]^2 + 4 sigma + jump reach 15):
  phi_0: R = 7.247e-13
  phi_1: R = 6.152e-13
  phi_2: R = 3.923e-13
  phi_3: R = 2.817e-13
  phi_4: R = 7.817e-13
  phi_5: R = 4.430e-13
max R = 7.817e-13


## The seven methods

All methods share one **taming policy**: the same map $b \mapsto b/(1+\Delta t\,\|b\|)$ is applied to every method's drift (ULA, the MALA proposal, FLA, the BAOAB force, raw CP, LSC-CP). Tamed MALA is still exact because the proposal density $q(y|x) = \mathcal N(y;\, x + \Delta t\, b_{\rm tamed}(x),\, 2\varepsilon\Delta t\, I)$ is used consistently in both directions of the MH ratio; asymmetric taming would make taming a hidden variable in the comparison.

**1. ULA.** $X \leftarrow X + \Delta t\,\mathrm{tame}(-\nabla V) + \sqrt{2\varepsilon\Delta t}\,\xi$.

**2. MALA.** With $\nabla\log\pi = -\beta\nabla V$, the proposal $Y = X - \tfrac{h\beta}2\nabla V(X) + \sqrt h\,\xi$ matches the ULA step iff $\tfrac{h\beta}{2} = \Delta t$ **and** $h = 2\varepsilon\Delta t$. Both conditions coincide:
$$\boxed{h = \frac{2\Delta t}{\beta} = 2\varepsilon\Delta t = \Delta t/4 \quad\text{at }\beta=8.}$$
Log-acceptance
$$\log\alpha = -\beta[V(Y)-V(X)] - \frac{1}{2h}\Big[\|X - \mu(Y)\|^2 - \|Y-\mu(X)\|^2\Big],\qquad \mu(z) = z + \Delta t\,\mathrm{tame}(-\nabla V(z)),$$
accepted elementwise. Proposals are **never clipped before the accept step** (that silently breaks exactness); out-of-box proposals are auto-rejected, which is valid MH for the box-restricted target. Expect acceptance $\approx 1$ — the honest message: **rejection does not cure metastability**.

**3. FLA / FLMC** (Şimşekli, ICML 2017, §3.3). $X \leftarrow X + \Delta t\,\mathrm{tame}(-c_\alpha \nabla U) + \Delta t^{1/\alpha}\,\xi^{(\alpha)}$ with $U = \beta V$, $c_\alpha = \Gamma(\alpha-1)/\Gamma(\alpha/2)^2$, $\alpha = 1.7$; per-coordinate $S\alpha S(1)$ noise by Chambers–Mallows–Stuck. **No tail clipping** — a truncated stable is not stable. FLA is the *uncorrected nonlocal* comparator: heavy tails cross barriers, but the invariant law is not $\pi$.

**4. Kinetic Langevin (BAOAB).** It is **not HMC** (no accept/reject; carries $O(\Delta t^2)$ configurational bias). Unit mass, $\gamma = 1$; the O-step coefficient is the exact OU solution: $dp = -\gamma p\,dt + \sqrt{2\gamma\varepsilon}\,dW$ gives $\mathrm{Var} = 2\gamma\varepsilon\int_0^{\Delta t}e^{-2\gamma s}ds = \varepsilon(1-e^{-2\gamma\Delta t})$ by Itô isometry. The trailing force is cached as the next step's leading B (one gradient per step).

**5. Parallel tempering.** MALA-within-replica at $\beta_k = \beta\, r^{k-1}$, replica $k$ using $h_k = 2\Delta t/\beta_k$ (same tamed drift step for every replica; only the noise scale differs). Adjacent swaps every $n_{\rm swap}$ steps (alternating parity); the joint target is $\prod_k \pi_k$ and the swap is a deterministic involution, so
$$\alpha_{\rm swap} = \min\Big\{1,\ \exp\big[(\beta_i - \beta_{i+1})\big(V(x_i) - V(x_{i+1})\big)\big]\Big\}.$$
$V$ values are cached by MALA, so swaps are free in evaluation count. $K$ is tuned so the mean swap acceptance lands in $[0.2, 0.4]$. The replica index is a batch dimension, state $(K, N, d)$; metrics use the cold replica only; **wall-clock includes all $K$ replicas**.

**6/7. Raw CP and LSC-CP.** Identical time discretisation,
$$X^{(1)} = X_n + \Delta t\,\frac{b(X_n)}{1+\Delta t\,\|b(X_n)\|} + \sqrt{2\varepsilon\Delta t}\;\xi_n,\qquad X_{n+1} = X^{(1)} + \sum_{k=1}^{N_n} A_k,$$
$N_n \sim \mathrm{Poisson}(\lambda\Delta t)$, $A_k \stackrel{iid}\sim \nu$; $b = -\nabla V$ (raw CP) or $b = -\nabla V + S_{\nu,\beta}$ (LSC-CP). The jump stream is a dedicated generator seeded identically for both methods, so their jump times and increments are **pathwise identical** (verified in `tests/test_samplers.py`), not merely equal in law.

Expect $K \approx 15$–$25$ replicas for this target — that *is* the finding: PT needs a long ladder to bridge $\beta\Delta V$ up to $\sim27$, and every replica costs a gradient per step.

In [5]:
# PT ladder: geometric in beta, K tuned so mean swap acceptance is in [0.2, 0.4]
gen = torch.Generator(device=DEV); gen.manual_seed(0)
x0_pilot = exp.init_fn(min(512, cfg.n_particles), gen)
pt_betas, ladder_info = tune_ladder(exp.pot, x0_pilot, cfg.dt, exp.box,
                                    C.BETA, exp.pt_beta_min, pilot_steps=600)
print(f"PT ladder: K={ladder_info['K']}  r={ladder_info['r']:.4f}  "
      f"beta_K={pt_betas[-1].item():.4f}  swap acceptance={ladder_info['swap_acceptance']:.3f}")
print("tuning history {K: acceptance}:", ladder_info["history"])

PT ladder: K=2  r=0.0625  beta_K=0.5000  swap acceptance=0.298
tuning history {K: acceptance}: {8: 0.8847005208333332, 6: 0.8374782986111107, 4: 0.73505859375, 3: 0.60107421875, 2: 0.2977213541666667}


## Reference, partition, metrics, bias floors

Reference sample size equals the run's $N$; metrics are evaluated at every checkpoint (cadence fixed in $t$, identical across methods).

* **$W_2$**: exact in 1D (sorted coupling); **sliced** $W_2$ for $d\ge2$ with $L=200$ projections drawn once from a fixed seed and reused across all times and methods (its bias floor decays like $N^{-1/2}$, not $N^{-1/d}$).
* **TV** (occupancy, on the partition): $\tfrac12\sum_k|\hat p_k - p^\star_k|$ — a **lower bound** on the full TV.
* **MMD**: Gaussian kernel, bandwidth **frozen once** by the median heuristic on the reference sample (per-frame bandwidths would make curves non-comparable); biased V-statistic $\widehat{\mathrm{MMD}}_b^2 = \|\mu_X-\mu_Y\|_{\mathcal H}^2 \ge 0$.
* **EMC** $= e^{H(\hat p)}/K$: plotted with a horizontal line at the target $e^{H(p^\star)}/K$; EMC $=1$ is optimal only for uniform $p^\star$, and deviation in *either* direction is error.
* **EJS**: base-2 Jensen–Shannon divergence between $\hat p$ and $p^\star$ (Blessing et al., arXiv:2406.07423, App. A.3), bounded in $[0,1]$, quadratic near the target, so it stays informative where TV saturates.
* **Bias floors** (mandatory): each metric between two independent reference samples of size $N$, 20 replicates; dashed line on every panel. Without this, every plateau is uninterpretable.
* **Nonfinite fraction**: logged per method per checkpoint; must be identically zero — metrics on survivors only would be survivorship bias, so nothing is ever filtered.

**Coverage vs correctness, once:** EMC measures *coverage*, TV/EJS measure *correctness*. Raw CP, whose invariant law is not $\pi$, over-flattens — driving EMC toward 1 (above its target line) while TV and EJS stay bad. That pairing *is* the raw-CP-vs-LSC-CP story.

Sanity check below: the frozen median-heuristic bandwidth must land near the mode spacing (several units), not the component width (1).

In [6]:
metrics_fn, floors, aux = make_metrics(exp, cfg.n_particles)
emc_target = exp.emc_target
print("p_star:", np.round(exp.p_star.cpu().numpy(), 6))
print("EMC target line: %.4f" % emc_target)
print("MMD bandwidth (median heuristic on reference, frozen):", round(aux["bandwidth"], 4))
print("bias floors (mean +- std over 20 replicate pairs):")
for k, v in floors.items():
    print(f"  {k:>12s}: {v['mean']:.5f} +- {v['std']:.5f}")
assert aux["bandwidth"] > 3.0, "bandwidth should reflect mode spacing, not component width"

p_star: [0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025
 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025
 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025
 0.025 0.025 0.025 0.025]
EMC target line: 1.0000
MMD bandwidth (median heuristic on reference, frozen): 41.1062
bias floors (mean +- std over 20 replicate pairs):
            W2: 1.30833 +- 0.27786
           MMD: 0.01558 +- 0.00538
            TV: 0.05091 +- 0.00605
           EJS: 0.00283 +- 0.00067
           EMC: 0.99221 +- 0.00183


## $\Delta t$ refinement and production

Declared $\Delta t$ selection rule, applied uniformly to every experiment (reported in the SI): **the largest $\Delta t$ on a dyadic grid at which every method's terminal value of every metric is within 5% of its $\Delta t/2$ value.** Three statistical guards make the rule meaningful at a single refinement seed: differences are measured relative to $\max(|m_{\Delta t/2}|,\ \text{bias floor})$; when *both* values sit inside the floor band (floor mean $+\,3$ s.d.) they are declared in agreement; and differences within $4\times$ the floor s.d. — the natural unit of single-run metric sampling noise at this $N$ — are likewise noise, not discretisation bias. The same guards apply to the quadrature-refinement comparison.

**One declared exception:** FLA does not gate the $\Delta t$ selection. Its continuum limit is not $\pi$, so its bias has no $\Delta t$ at which it should stabilise (empirically its density error *drifts monotonically* under refinement); demanding 5% stability from it would refine $\Delta t$ forever. FLA still runs at the shared chosen $\Delta t$, and its deviations across the dyadic grid are recorded in the refinement table for transparency.

Production protocol: 5 seeds $\times$ 7 methods, run **sequentially** (never batched) so per-run wall-clock is meaningful; all methods share $x_0$ per seed; 20 untimed warm-up steps absorb allocator/JIT effects; `torch.cuda.synchronize()` brackets every timed region, and the timer covers sampler work only.

In [7]:
def run_terminal_lsc(**quad):
    f = make_sampler_factory(exp, cfg.dt, pt_betas, score_kwargs=quad)
    n_ = int(round(cfg.T / cfg.dt))
    r_, _ = run_one("LSC-CP", 0, f, n_, n_, cfg.dt, metrics_fn, exp.pot, quiet=True)
    return {k: r_[-1][k] for k in ("W2", "TV", "MMD", "EMC", "EJS")}

settings = [dict(m_phi=m) for m in (16, 32, 64)]
CHOSEN_QUAD, quad_table = quadrature_refinement(
    settings, run_terminal_lsc, lambda **s: cert_e2(**s)["max_residual"], floors)
print("chosen production quadrature:", CHOSEN_QUAD)
display(pd.DataFrame(quad_table).round(6))
if CHOSEN_QUAD != DEFAULT_QUAD:
    cert_report = cert_e2(**CHOSEN_QUAD)
    print("certificate re-evaluated at chosen orders: max R =",
          f"{cert_report['max_residual']:.3e}")
    assert cert_report["max_residual"] < 1e-6

chosen production quadrature: {'m_phi': 16}


,m_phi,R,W2,TV,MMD,EMC,EJS,pass
0,16,0.0,3.118384,0.0638,0.052954,0.987323,0.004667,True
1,32,0.0,2.985960,0.0642,0.052213,0.987227,0.004646,True
2,64,0.0,3.031296,0.0650,0.053009,0.986888,0.004786,True


certificate re-evaluated at chosen orders: max R = 3.027e-08


In [8]:
MAIN_METRICS = ["W2", "TV", "MMD", "EMC", "EJS"]

def run_terminal_all(dt_):
    n_ = int(round(cfg.T / dt_))
    factory = make_sampler_factory(exp, dt_, pt_betas, score_kwargs=CHOSEN_QUAD)
    out = {}
    for m in C.METHODS:
        rows_, _ = run_one(m, 0, factory, n_, n_, dt_, metrics_fn, exp.pot, quiet=True)
        out[m] = {k: rows_[-1][k] for k in MAIN_METRICS}
    print(f"  refine_dt: finished pass at dt={dt_}", flush=True)
    return out

dt_final, dt_table = refine_dt(run_terminal_all, cfg.dt, floors, exclude=("FLA",))
print("chosen dt:", dt_final)
for row in dt_table:
    print(row)

n_steps = int(round(cfg.T / dt_final))
steps_per_ck = max(1, n_steps // C.N_CHECKPOINTS)
factory = make_sampler_factory(exp, dt_final, pt_betas, score_kwargs=CHOSEN_QUAD)
t0 = time.time()
rows, method_info = run_experiment(C.METHODS, cfg.seeds, factory, n_steps,
                                   steps_per_ck, dt_final, metrics_fn, exp.pot)
print(f"production total: {time.time()-t0:.0f}s")
worst_nonfinite = max(r["nonfinite_frac"] for r in rows)
assert worst_nonfinite == 0.0, worst_nonfinite
print("nonfinite fraction: identically zero across all methods/checkpoints")

  refine_dt: finished pass at dt=0.01


  refine_dt: finished pass at dt=0.005


chosen dt: 0.01
{'dt': 0.01, 'pass': True, 'failures': [], 'excluded_deviations': []}


  ULA seed 0: 1.7s sampler wall-clock


  ULA seed 1: 1.7s sampler wall-clock


...

  LSC-CP seed 2: 211.7s sampler wall-clock


  LSC-CP seed 3: 212.4s sampler wall-clock


  LSC-CP seed 4: 205.0s sampler wall-clock


LSC-CP: done in 1057.5s total


production total: 1215s
nonfinite fraction: identically zero across all methods/checkpoints


## Figures

In [9]:
fig_metrics = ("W2", "TV", "MMD", "EMC", "EJS")
written = make_all_figures(rows, FIGURES, floors, emc_target, metrics=fig_metrics)
print(f"{len(written)} figures x 3 formats (.pdf/.png 600dpi/.eps) + captions -> {FIGURES}")

# grid display for inspection (saved files above are one-figure-per-file)
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
n_m = len(fig_metrics)
fig, axes = plt.subplots(n_m, 2, figsize=(11, 3.2 * n_m))
for i, metric in enumerate(fig_metrics):
    for j, tag in enumerate(("vs_time", "vs_wallclock")):
        ax = axes[i, j] if n_m > 1 else axes[j]
        ax.imshow(mpimg.imread(os.path.join(FIGURES, f"{metric}_{tag}.png")))
        ax.set_axis_off()
plt.tight_layout(); plt.show()

10 figures x 3 formats (.pdf/.png 600dpi/.eps) + captions -> /home/zheyuanlai/levy-sampling/JCP_experiments/figures/mog40


## CSV emission, Hungarian spot-check, summary

Terminal exact-$W_2$ spot-check on a 500-point subsample (Hungarian algorithm), 2D only — a cross-validation of the sliced estimator, not a headline number.

In [10]:
gen_h = torch.Generator(device=DEV); gen_h.manual_seed(202)
ref_sub = exp.ref_sample(2500, gen_h)
from src.metrics import hungarian_w2
hungarian = {}
for m in C.METHODS:
    xf = method_info[m]["final_positions_seed0"]
    hungarian[m] = hungarian_w2(xf, ref_sub, m=500)
print("terminal Hungarian W2 (M=500 subsample, seed 0):")
for m, v in hungarian.items():
    print(f"  {m:>8s}: {v:.4f}")
ts_path = os.path.join(RESULTS, "metrics_timeseries.csv")
write_timeseries_csv(rows, ts_path)
summary_metrics = MAIN_METRICS + ["nonfinite_frac"]
summary = write_summary_csv(rows, C.METHODS, cfg.seeds, summary_metrics,
                            method_info, floors, os.path.join(RESULTS, "summary.csv"))

manifest = dict(
    experiment=EXPERIMENT,
    config=dict(d=cfg.d, N=cfg.n_particles, T=cfg.T, dt0=cfg.dt, dt=dt_final,
                beta=cfg.beta, eps=cfg.eps, lam=cfg.lam, seeds=list(cfg.seeds),
                n_checkpoints=C.N_CHECKPOINTS, warmup_steps=C.N_WARMUP_STEPS),
    quadrature=dict(chosen=CHOSEN_QUAD, table=quad_table),
    dt_refinement=[{k: (str(v) if isinstance(v, tuple) else v) for k, v in row.items()}
                   for row in dt_table],
    pt_ladder={k: v for k, v in ladder_info.items()},
    certificate=cert_report,
    bias_floors=floors,
    barrier_verification=barrier_report,
    method_info={m: {k: v for k, v in mi.items() if isinstance(v, (int, float))}
                 for m, mi in method_info.items()},
    hardware=hardware_manifest(),
    hungarian_w2_terminal=hungarian,
)
write_manifest(os.path.join(RESULTS, "manifest.json"), **manifest)
print("wrote", ts_path)
from IPython.display import display
display(pd.read_csv(os.path.join(RESULTS, "summary.csv")).round(5))

terminal Hungarian W2 (M=500 subsample, seed 0):
       ULA: 36.9535
      MALA: 36.9173
       FLA: 22.5968
     BAOAB: 37.1314
        PT: 28.5416
        CP: 15.1378
    LSC-CP: 6.4009


wrote /home/zheyuanlai/levy-sampling/JCP_experiments/results/mog40/metrics_timeseries.csv


,method,EJS_mean,EJS_std,EMC_mean,EMC_std,MMD_mean,MMD_std,TV_mean,TV_std,V_evals_per_step,...,jump_count_mean,m_clip_fraction,mala_accept,nonfinite_frac_mean,nonfinite_frac_std,pt_swap_accept,score_quad_evals_per_step,time_to_threshold_TV,wallclock_mean_s,wallclock_std_s
0,ULA,0.84581,0.00348,0.03369,0.00055,0.47703,0.00060,0.91728,0.00318,0.0,...,NaN,NaN,NaN,0.0,0.0,NaN,0.0,inf,1.71532,0.00619
1,MALA,0.84713,0.00267,0.03339,0.00062,0.47719,0.00035,0.91968,0.00303,2500.0,...,NaN,NaN,0.99994,0.0,0.0,NaN,0.0,inf,5.24202,0.10312
2,FLA,0.19839,0.00656,0.59039,0.01001,0.31950,0.00552,0.43760,0.00831,0.0,...,NaN,NaN,NaN,0.0,0.0,NaN,0.0,inf,2.81634,0.12221
3,BAOAB,0.85713,0.00275,0.03154,0.00050,0.47830,0.00046,0.93076,0.00397,0.0,...,NaN,NaN,NaN,0.0,0.0,NaN,0.0,inf,2.10376,0.00835
4,PT,0.38147,0.00464,0.36912,0.00442,0.39881,0.00107,0.60992,0.00544,5000.0,...,NaN,NaN,0.99985,0.0,0.0,0.19486,0.0,inf,5.54502,0.04129
5,CP,0.08299,0.00564,0.80067,0.01062,0.19085,0.00680,0.27904,0.01029,0.0,...,0.00997,NaN,NaN,0.0,0.0,NaN,0.0,inf,3.36730,0.00277
6,LSC-CP,0.00469,0.00115,0.98713,0.00318,0.04941,0.00597,0.06580,0.00977,0.0,...,0.00997,0.0,NaN,0.0,0.0,NaN,0.0,64.0,209.23854,3.94489
